**Nombre completo:** **REYNA VANESA ISAAC VIVANCO**

**Grupo:** **002**

**Fecha:** **12/10/2026**

---

# Experimentación con diferentes parámetros
**Simulación de Sistemas · Sesión del 7 de octubre de 2026 · Primera hora**

Hoy no vamos a aprender un método nuevo: vamos a **mover las perillas** de los que ya tenemos.

El método de hoy es de tres pasos:
1. Cambio un parámetro.
2. **Predigo** qué va a pasar (escríbelo en un comentario *antes* de correr).
3. Corro y comparo contra lo esperado.

Solo usamos `random` y `math`. Ejemplo de hilo: la ventanilla del banco.

In [2]:
import math
import random

random.seed(7)   # misma semilla, misma serie

## Funciones base
`exponencial(media)` es la transformada inversa de la primera hora. Todo lo demás la reutiliza.

Fíjate en el `1 - u`: `random.random()` puede dar 0, y el logaritmo de 0 truena. `1 - u` nunca es 0.

In [4]:
def exponencial(media):
    u = random.random()
    return -math.log(1 - u) * media


def entre_llegadas(lam):
    """lam = clientes por minuto. Devuelve minutos hasta el siguiente cliente."""
    return exponencial(1 / lam)


def promedio(f, n=20000):
    return sum(f() for _ in range(n)) / n

## Perilla 1: `lam` (llegadas por minuto) — inversa
**Predice:** si `lam` sube, ¿el tiempo entre llegadas sube o baja?

In [10]:
# Mi predicción: ...

for lam in (0.1, 0.25, 0.5, 1.0):
    m = promedio(lambda: entre_llegadas(lam))
    print(f"lam={lam}: promedio={m:.2f}  esperado={1/lam:.2f}")

lam=0.1: promedio=10.10  esperado=10.00
lam=0.25: promedio=4.02  esperado=4.00
lam=0.5: promedio=1.99  esperado=2.00
lam=1.0: promedio=1.00  esperado=1.00


**Ahora tú:** prueba con `lam = 2`. ¿Qué esperas? Escríbelo antes de correr.

In [14]:
# Mi predicción: Si aumento lam a 2, el tiempo promedio entre llegadas será menor (2 minutos) porque los clientes llegan más rápido.

for lam in (2.00, 0.5, 1.0):
    m = promedio(lambda: entre_llegadas(lam))
    print(f"lam={lam}: promedio={m:.2f} esperado={1/lam:.2f}")

lam=2.0: promedio=0.50 esperado=0.50
lam=0.5: promedio=1.99 esperado=2.00
lam=1.0: promedio=1.00 esperado=1.00


## Perilla 2: `k` (etapas de servicio) — convolución
Un servicio de `k` etapas, cada una de unos 2 minutos. Es un pipeline: un `for` que suma lo que tarda cada etapa.

**Predice:** si paso de 1 a 8 etapas, ¿cuánto crece el promedio?

In [18]:
def servicio_etapas(k, media_etapa=2.0):
    total = 0
    for _ in range(k):
        total += exponencial(media_etapa)
    return total


# Mi predicción: ...

for k in (1, 2, 4, 8):
    m = promedio(lambda: servicio_etapas(k))
    print(f"k={k}: promedio={m:.2f}  esperado={2*k}")

# Mi predicción: Si paso de 1 a 8 etapas, el promedio crecerá linealmente multiplicándose por 8, llegando a un valor esperado de 80.

for k in (2, 5, 44, 80):
    m = promedio(lambda: servicio_etapas(k))
    print(f"k={k}: promedio={m:.2f} esperado={2*k}")

k=1: promedio=2.00  esperado=2
k=2: promedio=3.97  esperado=4
k=4: promedio=8.01  esperado=8
k=8: promedio=15.99  esperado=16
k=2: promedio=3.99 esperado=4
k=5: promedio=9.92 esperado=10
k=44: promedio=88.02 esperado=88
k=80: promedio=159.97 esperado=160


## Perilla 3: `p` (clientes rápidos) — composición
Con probabilidad `p` el cliente es rápido (unos 2 min); si no, es lento (unos 8 min). Es un `if` al azar.

**Predice:** si bajo `p`, ¿el promedio sube o baja?

In [20]:
def servicio_mixto(p_rapido, media_rapido=2.0, media_lento=8.0):
    if random.random() < p_rapido:
        return exponencial(media_rapido)
    return exponencial(media_lento)


# Mi predicción: ...

for p in (0.9, 0.7, 0.5, 0.3):
    m = promedio(lambda: servicio_mixto(p))
    print(f"p={p}: promedio={m:.2f}  esperado={2*p + 8*(1-p):.2f}")


# Mi predicción: Si bajo la probabilidad p de clientes rápidos, habrá más clientes lentos, por lo que el tiempo promedio general de servicio va a subir.

for p in (0.9, 0.005, 0.25, 0.35):
    m = promedio(lambda: servicio_mixto(p))
    print(f"p={p}: promedio={m:.2f} esperado={2*p + 8*(1-p):.2f}")

p=0.9: promedio=2.60  esperado=2.60
p=0.7: promedio=3.85  esperado=3.80
p=0.5: promedio=5.07  esperado=5.00
p=0.3: promedio=6.13  esperado=6.20
p=0.9: promedio=2.61 esperado=2.60
p=0.005: promedio=8.06 esperado=7.97
p=0.25: promedio=6.59 esperado=6.50
p=0.35: promedio=5.92 esperado=5.90


## Perilla 4: `mu` y `sigma` — transformación directa
`random.gauss` ya trae la receta (por dentro usa Box-Müller). Tú solo pasas el promedio y la dispersión.

El `max(0.0, ...)` evita tiempos negativos: nadie tarda menos de cero minutos.

**Predice:** con el mismo `mu`, si `sigma` pasa de 1 a 3, ¿qué cambia: el promedio o qué tan lejos quedan los extremos?

In [22]:
def servicio_normal(mu, sigma):
    return max(0.0, random.gauss(mu, sigma))


# Mi predicción: ...

for mu, sigma in ((5, 1), (5, 3), (8, 1)):
    vals = [servicio_normal(mu, sigma) for _ in range(20000)]
    m = sum(vals) / len(vals)
    s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
    print(f"mu={mu} sigma={sigma}: media={m:.2f} desv={s:.2f}")


# Mi predicción: El promedio se mantendrá similar (cercano a 5), pero al aumentar sigma la dispersión (desviación estándar) será mayor, abarcando valores más lejanos.

for mu, sigma in ((5, 5), (4, 2), (5, 0.25)):
    vals = [servicio_normal(mu, sigma) for _ in range(20000)]
    m = sum(vals) / len(vals)
    s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
    print(f"mu={mu} sigma={sigma}: media={m:.2f} desv={s:.2f}")

mu=5 sigma=1: media=4.99 desv=1.00
mu=5 sigma=3: media=5.05 desv=2.86
mu=8 sigma=1: media=8.00 desv=1.00
mu=5 sigma=5: media=5.35 desv=4.30
mu=4 sigma=2: media=4.00 desv=1.94
mu=5 sigma=0.25: media=5.00 desv=0.25


> **Ojo:** pedimos `sigma = 3` y la desviación sale un poco menor (≈ 2.9). Es por el `max`: los tiempos negativos se cortan en cero y eso le quita dispersión.

## Perilla 5: cuántas corridas
Una **corrida** es una ejecución completa; una **réplica** es repetirla con otra semilla.

`corrida_ventanilla` simula 10 clientes y regresa la espera promedio. Tres "equipos" hacen distinto número de corridas.

**Predice:** si tres equipos hacen **una sola** corrida, ¿les sale lo mismo?

In [26]:
def corrida_ventanilla(n, lam, media_servicio):
    llegada = 0.0
    fin_anterior = 0.0
    esperas = []
    for i in range(n):
        if i > 0:
            llegada += entre_llegadas(lam)
        servicio = exponencial(media_servicio)
        inicio = max(llegada, fin_anterior)
        fin_anterior = inicio + servicio
        esperas.append(inicio - llegada)
    return sum(esperas) / n


# Mi predicción: ...

for n_corridas in (1, 5, 30, 1000):
    res = []
    for _ in range(3):  # tres "equipos" distintos
        esperas = [corrida_ventanilla(10, 0.25, 3.0) for _ in range(n_corridas)]
        res.append(sum(esperas) / n_corridas)
    print(n_corridas, [round(r, 2) for r in res])


# Mi predicción: No les saldrá lo mismo si hacen una sola corrida debido a la aleatoriedad, pero los resultados coincidirán más conforme aumentemos el número de corridas a 1000.

for n_corridas in (1, 5, 30, 1000):
    res = []
    for _ in range(3): # tres "equipos" distintos
        esperas = [corrida_ventanilla(10, 0.25, 3.0) for _ in range(n_corridas)]
        res.append(sum(esperas) / n_corridas)
    print(n_corridas, [round(r, 2) for r in res])

1 [0.21, 0.82, 0.38]
5 [2.73, 5.08, 1.92]
30 [3.46, 2.93, 2.94]
1000 [3.14, 3.16, 3.4]
1 [3.5, 0.68, 4.04]
5 [2.05, 2.47, 2.23]
30 [3.65, 3.04, 3.42]
1000 [3.29, 3.29, 3.19]


Con 1 corrida los tres equipos obtienen respuestas muy distintas. Con 1000, coinciden.

**Para pensar:** si un equipo hizo 1 corrida y otro hizo 1000, ¿a cuál le entregarías el número para decidir si abrir una segunda ventanilla?

## Actividad 7.1 — tu turno
1. Elige **una** perilla (`lam`, `k`, `p`, o `mu` y `sigma`).
2. **Predice** qué pasa y escríbelo en un comentario.
3. Corre con **cuatro valores** distintos y compara contra tu predicción.
4. Entrega este notebook (`.ipynb`) en tu repo de GitHub; el link va en la plataforma.

In [28]:
# Tu experimento aquí
# Mi predicción: Al experimentar con cuatro valores distintos de lam (0.225, 0.5, 1 y 2.05), el promedio del tiempo entre llegadas disminuirá de forma inversa conforme lam sea más grande.

for lam in (0.225, 0.5, 1, 2.05):
    m = promedio(lambda: entre_llegadas(lam))
    print(f"--- Experimento Actividad 7.1 --- lam={lam} -> Promedio obtenido: {m:.2f} (Valor esperado: {1/lam:.2f})")

--- Experimento Actividad 7.1 --- lam=0.225 -> Promedio obtenido: 4.45 (Valor esperado: 4.44)
--- Experimento Actividad 7.1 --- lam=0.5 -> Promedio obtenido: 2.00 (Valor esperado: 2.00)
--- Experimento Actividad 7.1 --- lam=1 -> Promedio obtenido: 1.01 (Valor esperado: 1.00)
--- Experimento Actividad 7.1 --- lam=2.05 -> Promedio obtenido: 0.49 (Valor esperado: 0.49)
